In [5]:
from src.authorization.auth import *
import geemap
import ee
from datetime import datetime, timezone, time

In [9]:
def get_image(coordinates: list[float],
                   image_collection: str,
                   start_date: str,
                   end_date: str,
                   cloud_percentage: float | int
                   ) -> tuple[str, ee.Geometry]:

    roi = ee.Geometry.Point(coordinates).buffer(450).bounds()

    image = (
        ee.ImageCollection(image_collection)
        .filterBounds(roi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', cloud_percentage))
        .sort('CLOUDY_PIXEL_PERCENTAGE')
        .first()
    )
    full_id = image.get('system:id').getInfo()

    return full_id, roi


def get_metadata(image_id: str) -> dict[str, str]:
    img = ee.Image(image_id)
    props = img.toDictionary().getInfo()

    time_start = props.get('system:time_start')
    acquired_at = (
        datetime.fromtimestamp(time_start/1000, tz=timezone.utc).isoformat()
        if time_start else None
    )
    return props
    #return {
    #    "system_id": image_id,
    #    "acquired_at": acquired_at,
    #    "cloud_pct": props.get("CLOUDY_PIXEL_PERCENTAGE"),
    #    "mgrs_tile": props.get("MGRS_TILE"),
    #    "product_id": props.get("PRODUCT_ID"),
    #    "platform": props.get("SPACECRAFT_NAME"),
    #    "processing_baseline": props.get("PROCESSING_BASELINE"),
    #}

In [10]:
credentials = authenticate_google_api()
initialize_earth_engine(credentials)


data_to_find = {
    "roi_coordinates":[22.229681, 50.554120],
    "collection":'COPERNICUS/S2_SR_HARMONIZED',
    "start_date":'2024-09-01',
    "end_date":'2024-10-31',
    "cloud_percentage": 20
}
img_id, roi = get_image(coordinates=data_to_find.get('roi_coordinates'),
                        image_collection=data_to_find.get("collection"),
                        start_date=data_to_find.get("start_date"),
                        end_date=data_to_find.get("end_date"),
                        cloud_percentage=data_to_find.get("cloud_percentage"),
                        )
img_metadata = get_metadata(img_id)
print(img_metadata)

Earth Engine initialized successfully!
{'AOT_RETRIEVAL_ACCURACY': 0, 'AOT_RETRIEVAL_METHOD': 'SEN2COR_DDV', 'BOA_ADD_OFFSET_B1': -1000, 'BOA_ADD_OFFSET_B10': -1000, 'BOA_ADD_OFFSET_B11': -1000, 'BOA_ADD_OFFSET_B12': -1000, 'BOA_ADD_OFFSET_B2': -1000, 'BOA_ADD_OFFSET_B3': -1000, 'BOA_ADD_OFFSET_B4': -1000, 'BOA_ADD_OFFSET_B5': -1000, 'BOA_ADD_OFFSET_B6': -1000, 'BOA_ADD_OFFSET_B7': -1000, 'BOA_ADD_OFFSET_B8': -1000, 'BOA_ADD_OFFSET_B8A': -1000, 'BOA_ADD_OFFSET_B9': -1000, 'CAST_SHADOW_PERCENTAGE': 0, 'CLOUDY_PIXEL_OVER_LAND_PERCENTAGE': 0.00229, 'CLOUDY_PIXEL_PERCENTAGE': 0.002273, 'CLOUD_COVERAGE_ASSESSMENT': 0.002273, 'CLOUD_SHADOW_PERCENTAGE': 0.000186, 'DATASTRIP_ID': 'S2B_OPER_MSI_L2A_DS_2BPS_20240920T122840_S20240920T094449_N05.11', 'DATATAKE_IDENTIFIER': 'GS2B_20240920T094029_039387_N05.11', 'DATATAKE_TYPE': 'INS-NOBS', 'DEGRADED_MSI_DATA_PERCENTAGE': 0.0018, 'FORMAT_CORRECTNESS': 'PASSED', 'GENERAL_QUALITY': 'PASSED', 'GENERATION_TIME': 1726835320000, 'GEOMETRIC_QUALITY': 'PASSE

In [21]:

from datetime import timezone, datetime
generation_time = img_metadata.get('GENERATION_TIME')
dt = datetime.fromtimestamp(img_metadata.get('GENERATION_TIME')/1000, tz=timezone.utc).isoformat()
print(dt)

2024-09-20 12:28:40+00:00


In [ ]:
generation_time = 1726835320000

In [15]:
from datetime import timezone, datetime

dt = datetime.fromtimestamp(generation_time/1000, tz=timezone.utc)


#generation_date = date.fromtimestamp(1726835320000)
#print(generation_date)

2024-09-20
